# 001 - Baseline modeling dataset (short-horizon prediction)\n
\n
Цель: построить modeling dataset из реконструированного order book на сетке 100ms.\n
\n
Ноутбук — тонкий оркестратор:\n
- загрузка parquet за заданный период\n
- sanity checks (валидация состояний)\n
- сбор features + multi-horizon targets через модули\n
- сохранение dataset parquet и metadata\n
\n
Важно: вся вычислительная логика вынесена в Python-модули под `research/`, а не в монолитные ячейки.

## Leakage safeguards\n
\n
Правило: **в feature set попадают только значения, доступные на момент наблюдения t**.\n
- history-вычисления — это только `shift(+)` и rolling на прошлом (без будущего)\n
- labels-вычисления — это только будущие значения (через shift(-k) / aligned horizons)\n
- перед обучением мы drop-производим строки с NaN в target-колонках\n
\n
Для горизонтов 150/250ms и т.п. используется policy: **первое наблюдение на сетке, которое попадает в t + H (ceil до ближайшей grid-точки)**.\n
Это явно отражается в metadata.

In [ ]:
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd

# Root detection
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
]
root = next((p for p in candidates if (p / "pyproject.toml").exists()), Path.cwd())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from research.utils.reconstructed_parquet_loader import load_reconstructed_grid_parquet
from research.utils.dataset_assembly import ModelingDatasetConfig, build_modeling_dataset
from research.labels.profitability_targets import MakerTakerFeeModel

print("root:", root)

In [ ]:
# =============================
# CONFIG
# =============================
INST_ID = "BTC-USDT-SWAP"
START = "2026-03-23 00:00:00"
END = "2026-03-25 23:59:59"

# Dataset building parameters
GRID_MS = 100
DEPTH_K = 10

HORIZONS_MS = [100, 150, 200, 250, 500, 1000]
TICK_THRESHOLDS = (1, 2, 3)
LOOKBACK_MS = (100, 200, 500, 1000)

# Profitability-aware proxy
FEE_MODEL = MakerTakerFeeModel(
    maker_fee_rate=0.0002,  # пример: 2 bps
    taker_fee_rate=0.0007,  # пример: 7 bps
    min_profit_ticks=0.0,
)
GROSS_PROFIT_MIN_TICKS = 1.0

# TP/SL labeling
TAKE_PROFIT_TICKS = (1, 2, 3)
STOP_LOSS_TICKS = (1, 2)
HOLDING_MS = (100, 250, 500, 1000)

# Tick size: infer from data by default (safer than hardcoding)
TICK_SIZE = None  # или float, например 0.1

OUTPUT_DIR = root / "data" / "processed" / "modeling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("INST_ID:", INST_ID)
print("Window:", START, "..", END)

In [ ]:
start_ts = pd.Timestamp(START, tz="UTC")
end_ts = pd.Timestamp(END, tz="UTC")
base_dir = root / "data" / "reconstructed"

load_res = load_reconstructed_grid_parquet(
    base_dir=base_dir,
    inst_id=INST_ID,
    start_ts=start_ts,
    end_ts=end_ts,
    grid_dirname="grid100ms",
)
df = load_res.df

print("Loaded rows:", len(df))
print("Selected parquet files:", len(load_res.selected_files))
df.head()

In [ ]:
# =============================
# Sanity checks (no leakage)
# =============================
work = df.copy()
work["ts_event"] = pd.to_datetime(work["ts_event"], utc=True, errors="coerce")
work = work.dropna(subset=["ts_event"]).sort_values("ts_event").reset_index(drop=True)

quality = {}
quality["rows_before"] = len(work)
quality["duplicate_ts_rows"] = int(work.duplicated(subset=["ts_event"]).sum())

# negative/zero sizes
bid_sz_cols = [f"bid_sz_{i:02d}" for i in range(1, DEPTH_K + 1)]
ask_sz_cols = [f"ask_sz_{i:02d}" for i in range(1, DEPTH_K + 1)]
quality["negative_or_zero_sizes_total"] = int(((work[bid_sz_cols + ask_sz_cols] <= 0).sum().sum()))

# crossed book
quality["crossed_book_rows"] = int(((work["bid_px_01"] >= work["ask_px_01"]).fillna(False)).sum())

# filter invalid rows
mask_valid = (
    (work[bid_sz_cols + ask_sz_cols] > 0).all(axis=1)
    & (work["bid_px_01"] < work["ask_px_01"])
)
work = work.loc[mask_valid].copy()

quality["rows_after"] = len(work)

print("Quality report:\n", json.dumps(quality, indent=2, default=str))
work.head()

In [ ]:
# =============================
# Build dataset (features + labels)
# =============================
config = ModelingDatasetConfig(
    grid_ms=GRID_MS,
    depth_k=DEPTH_K,
    horizons_ms=tuple(HORIZONS_MS),
    tick_thresholds=TICK_THRESHOLDS,
    lookback_ms=LOOKBACK_MS,
    gross_profit_min_ticks=float(GROSS_PROFIT_MIN_TICKS),
    fee_model=FEE_MODEL,
    take_profit_ticks=TAKE_PROFIT_TICKS,
    stop_loss_ticks=STOP_LOSS_TICKS,
    holding_ms=HOLDING_MS,
    tick_size=TICK_SIZE,
)

dataset_df, metadata = build_modeling_dataset(work, config=config, ts_col="ts_event")

print("Dataset rows:", len(dataset_df))
print("Feature+target cols:", len(dataset_df.columns))
dataset_df.head()

In [ ]:
# =============================
# Target/feature lists + class balance
# =============================
target_cols = [
    c
    for c in dataset_df.columns
    if c.startswith("target_")
    or c.startswith("label_")
    or c.startswith("mid_move_ticks_")
    or c.startswith("best_bid_move_ticks_")
    or c.startswith("best_ask_move_ticks_")
]
feature_cols = [c for c in dataset_df.columns if c not in set(target_cols)]

print("#feature cols:", len(feature_cols))
print("#target cols:", len(target_cols))
metadata["targets"]

# Class balance for binary targets (skip {-1,0,1} labels)
binary_targets = [c for c in target_cols if dataset_df[c].dropna().unique().size <= 2 and dataset_df[c].dropna().dtype != object]

bal_rows = []
for c in binary_targets:
    s = dataset_df[c].value_counts(normalize=True)
    bal_rows.append({"target": c, "p1": float(s.get(1, 0.0)), "p0": float(s.get(0, 0.0))})

bal_df = pd.DataFrame(bal_rows).sort_values("target")
display(bal_df.head(30))
print("Binary targets total:", len(bal_df))

In [ ]:
# =============================
# Sanity plots on samples
# =============================
sample = dataset_df
if len(sample) > 300_000:
    sample = sample.sample(300_000, random_state=42)

for h in HORIZONS_MS:
    col = f"mid_move_ticks_h{h}"
    if col in sample.columns:
        ax = sample[col].hist(bins=80, figsize=(8, 3))
        ax = ax
        import matplotlib.pyplot as plt
        plt.title(col)
        plt.show()

print("Done sanity plots.")

In [ ]:
# =============================
# Save dataset + metadata
# =============================
run_tag = f"{pd.Timestamp(START).strftime('%Y%m%d_%H%M%S')}__{pd.Timestamp(END).strftime('%Y%m%d_%H%M%S')}" 
out_parquet = OUTPUT_DIR / f"{INST_ID.lower().replace('-', '_')}_model_dataset_{GRID_MS}ms_baseline_{run_tag}.parquet"
out_meta = OUTPUT_DIR / f"{INST_ID.lower().replace('-', '_')}_model_dataset_{GRID_MS}ms_baseline_{run_tag}_metadata.json"

dataset_df.to_parquet(out_parquet, index=False)

metadata["feature_cols"] = feature_cols
metadata["target_cols"] = target_cols
metadata["quality"] = quality
metadata["output_parquet"] = str(out_parquet)
metadata["output_metadata"] = str(out_meta)

out_meta.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")

print("Saved:")
print(out_parquet)
print(out_meta)

## Что считать готовым для следующего шага

- Dataset parquet сохранён в `data/processed/modeling/`
- В metadata есть tick size, aligned horizon policy и список `feature_cols` / `target_cols`
- Наличие target families:
  - raw directional (на mid)
  - tick-threshold binary directionality
  - gross/net profitability (maker-taker proxy)
  - TP/SL ordering label

Следующий шаг — обучить baseline model и использовать metadata для воспроизводимости.

## Принятые допущения (коротко)

- **Tick size**: по умолчанию инферится из `mid_px` (мода малых положительных изменений). При желании можно задать `TICK_SIZE` явно в конфиге.
- **Alignment горизонтов**: для `h=150/250ms` и т.п. на сетке 100ms используется policy **ceil-to-next-grid**: метка считается по первому доступному наблюдению, которое попадает в `t + H` (на регулярной сетке это соответствует шагам `ceil(H / grid_ms)`).
- **Profitability proxy (maker-taker)**:
  - Long: entry `best_bid_px_01(t)`, exit `best_ask_px_01(t+H)`.
  - Short: entry `best_ask_px_01(t)`, exit `best_bid_px_01(t+H)`.
  - Net profitability учитывает maker/taker fees как доли notional, плюс `min_profit_ticks`.
- **TP/SL labeling**:
  - TP/SL проверяется на **mid_px** относительно цены entry.
  - Long entry: `best_bid_px_01(t)`; TP — mid вверх на `TP*tick_size`, SL — mid вниз на `SL*tick_size`.
  - Short entry: `best_ask_px_01(t)`; TP — mid вниз на `TP*tick_size`, SL — mid вверх на `SL*tick_size`.
